# Task 1: News Exploratory Data Analysis

This notebook covers descriptive statistics for headline lengths, article counts by publisher, publisher/domain analysis, publication-frequency time series, spike discussion, and TF-IDF keyword/topic analysis.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from src.data_loading import load_news_data
from src.news_analysis import add_headline_features, add_publisher_domain, daily_publication_counts, spike_days

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load News Data

The full file has about 1.4M analyst-rating headlines. During early exploration, set `NROWS` to a smaller number if memory is limited; final analysis can set it to `None`.

In [ ]:
NROWS = None
news = load_news_data(nrows=NROWS)
news = add_headline_features(add_publisher_domain(news))

display(news.head())
news.info()

In [ ]:
headline_stats = news[['headline_char_count', 'headline_word_count']].describe().T
publisher_counts = news['publisher'].value_counts().rename_axis('publisher').reset_index(name='article_count')
stock_counts = news['stock'].value_counts().rename_axis('stock').reset_index(name='article_count')

display(headline_stats)
display(publisher_counts.head(15))
display(stock_counts.head(15))

## Publisher Analysis

This section identifies the most active sources and extracts domains when publishers are stored in email format.

In [ ]:
email_publishers = news.dropna(subset=['publisher_domain'])
domain_counts = email_publishers['publisher_domain'].value_counts().rename_axis('domain').reset_index(name='article_count')

print(f"Email-format publisher rows: {len(email_publishers):,}")
display(domain_counts.head(20))

In [ ]:
top_publishers = publisher_counts.head(15)
ax = sns.barplot(data=top_publishers, y='publisher', x='article_count', color='#3A7CA5')
ax.set_title('Top 15 Publishers by Article Count')
ax.set_xlabel('Article Count')
ax.set_ylabel('Publisher')
plt.tight_layout()
plt.show()

## Publication Frequency and Spike Days

Publication spikes are flagged with a z-score rule. Spikes usually represent market-wide news events, earnings cycles, analyst-action clusters, or broader volatility periods that trigger many short headlines.

In [ ]:
daily_counts = daily_publication_counts(news)
spikes = spike_days(daily_counts, z_threshold=2.0)

ax = daily_counts.plot(color='#2F4858', linewidth=1.2)
ax.scatter(pd.to_datetime(spikes.index), spikes.values, color='#F26419', label='Spike days')
ax.set_title('Daily Financial News Publication Frequency')
ax.set_xlabel('Publication Date')
ax.set_ylabel('Article Count')
ax.legend()
plt.tight_layout()
plt.show()

display(spikes.head(15).rename('article_count').reset_index().rename(columns={'index': 'publish_date'}))

## Keyword and Topic Analysis

TF-IDF is used to surface recurring financial themes while reducing the influence of very common words. CountVectorizer bigrams are also useful for detecting analyst-action phrases such as price target, earnings guidance, and stock moving.

In [ ]:
sample_headlines = news['headline'].dropna().astype(str)
if len(sample_headlines) > 250_000:
    sample_headlines = sample_headlines.sample(250_000, random_state=42)

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=25, max_features=60)
matrix = tfidf.fit_transform(sample_headlines)
scores = matrix.mean(axis=0).A1
keywords = pd.DataFrame({'term': tfidf.get_feature_names_out(), 'tfidf_score': scores})
keywords = keywords.sort_values('tfidf_score', ascending=False).head(25)
display(keywords)

In [ ]:
ax = sns.barplot(data=keywords.head(20), y='term', x='tfidf_score', color='#6A994E')
ax.set_title('Top TF-IDF Financial News Keywords and Phrases')
ax.set_xlabel('Mean TF-IDF Score')
ax.set_ylabel('Keyword / Phrase')
plt.tight_layout()
plt.show()

In [ ]:
count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(2, 2), min_df=20, max_features=40)
counts = count_vectorizer.fit_transform(sample_headlines)
bigram_counts = pd.DataFrame({
    'bigram': count_vectorizer.get_feature_names_out(),
    'count': counts.sum(axis=0).A1,
}).sort_values('count', ascending=False).head(20)

ax = sns.barplot(data=bigram_counts, y='bigram', x='count', color='#BC4749')
ax.set_title('Most Frequent Headline Bigrams')
ax.set_xlabel('Count')
ax.set_ylabel('Bigram')
plt.tight_layout()
plt.show()

## Recurring Financial Themes

The highest-weighted terms and bigrams are expected to cluster around analyst actions, earnings, price targets, market movers, highs/lows, upgrades, downgrades, and sector-specific catalysts. These themes are directly relevant for later sentiment work because they often encode market expectations and investor reaction in a compact headline format.

The publication-frequency spikes should be reviewed alongside the top stocks and publishers for those dates. A single spike can be caused by broad market volatility, earnings seasons, or syndicated publisher behavior, so spike days are a signal for follow-up analysis rather than proof of a causal price event.